# 1.1 量化（Quantization）

## 什么是量化？

量化是将模型中高精度浮点数（FP32/FP16）映射到低精度整数表示（INT8/INT4/INT2等）的技术。具体来说，量化通过一个线性映射函数，将连续的浮点值域离散化为有限的整数级别，从而大幅减少模型的存储空间和计算开销。

## 为什么用量化？

1. **存储压缩**：FP32权重占4字节，INT8占1字节，INT4仅占0.5字节，压缩比分别达4×和8×。端侧设备存储有限，量化使大模型部署成为可能。
2. **计算加速**：整数乘加运算（INT8 MAC）比浮点运算快2-4×，且NPU/DSP等端侧芯片通常只支持低精度整数运算。
3. **内存带宽降低**：推理的瓶颈往往是内存带宽而非计算，量化后数据搬运量减少，可显著降低延迟。
4. **功耗降低**：整数运算的功耗远低于浮点运算，对电池供电的端侧设备尤为重要。

## 量化的数学原理

量化的核心是一个仿射映射（affine mapping），将浮点数域 $[x_{min}, x_{max}]$ 线性映射到整数域 $[q_{min}, q_{max}]$：

$$x_q = \text{round}\left(\frac{x}{s}\right) + z$$

$$x_{deq} = (x_q - z) \times s$$

其中：
- $x$：原始浮点值
- $x_q$：量化后的整数值
- $x_{deq}$：反量化恢复的浮点值
- $s$：缩放因子（scale），表示每个量化级别对应的浮点步长
- $z$：零点（zero-point），浮点0对应的整数值
- $\text{round}(\cdot)$：四舍五入函数，引入量化噪声

量化误差来源于 $\text{round}$ 操作的舍入和 $\text{clamp}$ 操作的截断，理论上量化误差的方差为 $\frac{s^2}{12}$（均匀量化假设下）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import math

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 1.1.1 训练后量化（Post-Training Quantization, PTQ）

### 什么是PTQ？

训练后量化（PTQ）是在模型训练完成后，无需重新训练，直接将浮点模型转换为低精度整数模型的技术。它通过少量校准数据（calibration data）统计权重和激活值的数值分布，计算最优的量化参数（scale和zero-point），使量化误差最小化。

### 为什么用PTQ？

1. **零训练成本**：不需要反向传播和训练数据，只需少量前向推理即可完成量化，适合快速部署。
2. **通用性强**：适用于任何已训练好的模型，无需修改训练流程。
3. **效果显著**：INT8 PTQ通常仅损失不到1%的精度，却能获得4×存储压缩和2-4×推理加速。

### PTQ的原理

PTQ的核心流程：
1. **校准（Calibration）**：用少量代表性数据（通常128-512个样本）前向传播，收集各层激活值的统计信息（min、max、分布）。
2. **参数计算**：根据统计信息计算每层的scale和zero-point。
3. **量化转换**：将浮点权重转换为整数，存储scale和zero-point用于推理时反量化。

### 量化基本公式

**量化（Float → Int）：**
$$x_q = \text{round}(x / s) + z$$

**反量化（Int → Float）：**
$$x_{deq} = (x_q - z) \times s$$

其中：
- $x$：原始浮点值
- $x_q$：量化后的整数值
- $s$：缩放因子（scale），$s = \frac{x_{max} - x_{min}}{q_{max} - q_{min}}$，表示每个量化步长对应的浮点范围
- $z$：零点（zero-point），$z = \text{round}(q_{min} - x_{min}/s)$，确保浮点0被精确表示
- $q_{min}, q_{max}$：量化整数的取值范围，如INT8为$[-128, 127]$，UINT8为$[0, 255]$

注意：之所以先缩放后平移只是单纯因为硬件算子效率高的原因，而不是为了量化误差最小。先平移后缩放是理论上误差更小，但是硬件算子效率低，不推荐使用。

### 对称量化（Symmetric Quantization）

#### 什么是对称量化？

对称量化是量化范围关于零点对称的量化方式，zero-point固定为0，仅使用scale参数。量化后的整数范围对称分布在0两侧，如INT8的范围为$[-127, +127]$（注意不是$[-128, 127]$，因为要保证对称性）。

#### 为什么用对称量化？

1. **计算高效**：zero-point为0，省去了反量化时的减法操作，硬件实现更简单，推理速度更快。
2. **权重友好**：神经网络的权重通常近似服从零均值对称分布（如正态分布），对称量化能很好地拟合这种分布。
3. **内存节省**：只需存储一个scale参数，无需存储zero-point。

#### 对称量化的数学公式

**Scale计算：**
$$s = \frac{\max(|x|)}{2^{b-1}-1}$$

**量化：**
$$x_q = \text{clamp}(\text{round}(x / s), -2^{b-1}+1, 2^{b-1}-1)$$

**反量化：**
$$x_{deq} = x_q \times s$$

其中：
- $s$：缩放因子，由张量绝对值最大值决定
- $b$：量化比特数（如8表示INT8）
- $2^{b-1}-1$：量化级别的最大正值（如INT8为127）
- $\max(|x|)$：张量中绝对值最大的元素，决定了量化的动态范围

**局限性**：当数据分布偏斜时（如ReLU后的激活值全为正），对称量化会浪费一半的量化级别在永远不会出现的负数区域。

In [ ]:
def symmetric_quantize(tensor: torch.Tensor, n_bits: int = 8) -> tuple:
    """对称量化：zero-point固定为0"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    scale = tensor.abs().max() / q_max
    scale = torch.clamp(scale, min=1e-8)
    quantized = torch.clamp(torch.round(tensor / scale), q_min, q_max).to(torch.int8 if n_bits == 8 else torch.int32)
    return quantized, scale

def symmetric_dequantize(quantized: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """对称反量化"""
    return quantized.float() * scale

weight = torch.randn(128, 256) * 0.5
q_weight, scale = symmetric_quantize(weight, n_bits=8)
deq_weight = symmetric_dequantize(q_weight, scale)

mse = F.mse_loss(weight, deq_weight)
cos_sim = F.cosine_similarity(weight.unsqueeze(0), deq_weight.unsqueeze(0)).mean()

print(f"=== 对称量化 (INT8) ===")
print(f"原始权重大小: {weight.numel() * 4 / 1024:.2f} KB (FP32)")
print(f"量化权重大小: {q_weight.numel() * 1 / 1024:.2f} KB (INT8)")
print(f"压缩比: {4/1:.1f}x")
print(f"MSE: {mse:.6f}")
print(f"余弦相似度: {cos_sim:.6f}")
print(f"Scale: {scale:.6f}")
print(f"量化值范围: [{q_weight.min()}, {q_weight.max()}]")

### 非对称量化（Asymmetric Quantization）

#### 什么是非对称量化？

非对称量化允许量化范围不对称，使用独立的scale和zero-point两个参数。zero-point是一个整数偏移量，使得浮点0可以被精确表示为某个整数值，从而充分利用整个量化范围。

#### 为什么用非对称量化？

1. **拟合偏斜分布**：ReLU后的激活值全为正，非对称量化将全部$2^b$个量化级别用于$[0, x_{max}]$，而对称量化只能用一半的级别，精度损失显著。
2. **更高精度**：对于非对称分布，非对称量化的量化步长更小，舍入误差更低。
3. **激活值首选**：实际部署中，权重通常用对称量化，激活值通常用非对称量化。

#### 非对称量化的数学公式

**Scale计算：**
$$s = \frac{x_{max} - x_{min}}{2^b - 1}$$

**Zero-point计算：**
$$z = \text{round}(-x_{min} / s)$$

**量化：**
$$x_q = \text{clamp}(\text{round}(x / s) + z, 0, 2^b - 1)$$

**反量化：**
$$x_{deq} = (x_q - z) \times s$$

其中：
- $s$：缩放因子，由数据范围和量化级别数共同决定
- $z$：零点，确保浮点0映射到整数$z$，使得$0 \rightarrow z \rightarrow 0$精确可逆
- $x_{min}, x_{max}$：校准数据中该张量的最小值和最大值
- $2^b - 1$：量化级别的最大值（如UINT8为255）

**代价**：非对称量化在矩阵乘法中需要额外处理zero-point的交叉项：$\sum (x_{q,i} - z_x)(w_{q,i} - z_w) = \sum x_{q,i} w_{q,i} - z_w \sum x_{q,i} - z_x \sum w_{q,i} + z_x z_w \cdot n$，后三项需要预计算。

In [ ]:
def asymmetric_quantize(tensor: torch.Tensor, n_bits: int = 8) -> tuple:
    """非对称量化：使用独立的scale和zero-point"""
    q_max = 2 ** n_bits - 1
    q_min = 0
    x_min = tensor.min()
    x_max = tensor.max()
    scale = (x_max - x_min) / q_max
    scale = torch.clamp(scale, min=1e-8)
    zero_point = torch.round(-x_min / scale).clamp(q_min, q_max).to(torch.uint8 if n_bits == 8 else torch.int32)
    quantized = torch.clamp(torch.round(tensor / scale) + zero_point.float(), q_min, q_max).to(torch.uint8 if n_bits == 8 else torch.int32)
    return quantized, scale, zero_point

def asymmetric_dequantize(quantized: torch.Tensor, scale: torch.Tensor, zero_point: torch.Tensor) -> torch.Tensor:
    """非对称反量化"""
    return (quantized.float() - zero_point.float()) * scale

activation = F.relu(torch.randn(128, 256))
q_act, scale_a, zp_a = asymmetric_quantize(activation, n_bits=8)
deq_act = asymmetric_dequantize(q_act, scale_a, zp_a)

mse_a = F.mse_loss(activation, deq_act)
cos_sim_a = F.cosine_similarity(activation.unsqueeze(0), deq_act.unsqueeze(0)).mean()

print(f"=== 非对称量化 (UINT8) - ReLU后激活 ===")
print(f"原始激活范围: [{activation.min():.4f}, {activation.max():.4f}]")
print(f"MSE: {mse_a:.6f}")
print(f"余弦相似度: {cos_sim_a:.6f}")
print(f"Scale: {scale_a:.6f}, Zero-point: {zp_a}")

q_act_sym, scale_sym = symmetric_quantize(activation, n_bits=8)
deq_act_sym = symmetric_dequantize(q_act_sym, scale_sym)
mse_sym = F.mse_loss(activation, deq_act_sym)
print(f"\n对比 - 对称量化MSE: {mse_sym:.6f} vs 非对称量化MSE: {mse_a:.6f}")
print(f"非对称量化在偏斜分布(如ReLU后)上精度更优")

### 逐通道量化（Per-Channel Quantization）

#### 什么是逐通道量化？

逐通道量化为权重张量的每个输出通道独立计算量化参数（scale和zero-point），而不是整个张量共享一组参数。对于形状为$[C_{out}, C_{in}]$的权重矩阵，逐通道量化会产出$C_{out}$组独立的scale值。

#### 为什么用逐通道量化？

1. **通道间差异大**：不同输出通道的权重分布差异可能很大（某些通道数值范围大，某些很小），逐张量量化会被最大值主导，导致小值通道的量化精度极差。
2. **显著提升低比特精度**：INT4/INT2量化时，逐通道量化相比逐张量量化可降低50%以上的量化误差。
3. **硬件支持**：主流NPU/TPU均支持逐通道scale，额外开销仅为存储$C_{out}$个scale值。

#### 逐通道量化的数学公式

对于权重矩阵 $W \in \mathbb{R}^{C_{out} \times C_{in}}$，沿输出通道维度（axis=0）独立计算：

$$s_c = \frac{\max(|W_{c,:}|)}{2^{b-1}-1}, \quad c = 0, 1, ..., C_{out}-1$$

$$W_{q,c,:} = \text{clamp}(\text{round}(W_{c,:} / s_c), q_{min}, q_{max})$$

其中：
- $s_c$：第$c$个输出通道的缩放因子
- $W_{c,:}$：权重矩阵第$c$行的所有元素
- $W_{q,c,:}$：第$c$行量化后的整数值
- $q_{min}, q_{max}$：量化整数的取值范围

In [ ]:
def per_channel_quantize(tensor: torch.Tensor, n_bits: int = 8, axis: int = 0) -> tuple:
    """逐通道对称量化：沿指定轴每个通道独立计算scale"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    scale = tensor.abs().amax(dim=axis, keepdim=True) / q_max
    scale = torch.clamp(scale, min=1e-8)
    quantized = torch.clamp(torch.round(tensor / scale), q_min, q_max).to(torch.int8)
    return quantized, scale.squeeze(axis)

def per_channel_dequantize(quantized: torch.Tensor, scale: torch.Tensor, axis: int = 0) -> torch.Tensor:
    """逐通道反量化"""
    if axis == 0:
        scale = scale.unsqueeze(1)
    else:
        scale = scale.unsqueeze(0)
    return quantized.float() * scale

weight = torch.randn(128, 256)
weight[0:10] *= 10.0
weight[60:70] *= 0.01

q_per_tensor, scale_pt = symmetric_quantize(weight, n_bits=8)
deq_per_tensor = symmetric_dequantize(q_per_tensor, scale_pt)

q_per_channel, scale_pc = per_channel_quantize(weight, n_bits=8, axis=0)
deq_per_channel = per_channel_dequantize(q_per_channel, scale_pc, axis=0)

mse_per_tensor = F.mse_loss(weight, deq_per_tensor)
mse_per_channel = F.mse_loss(weight, deq_per_channel)

print(f"=== 逐通道 vs 逐张量量化 (INT8) ===")
print(f"逐张量 MSE: {mse_per_tensor:.6f}")
print(f"逐通道 MSE: {mse_per_channel:.6f}")
print(f"逐通道量化误差降低: {(1 - mse_per_channel/mse_per_tensor)*100:.1f}%")
print(f"\n逐张量 scale: {scale_pt:.6f}")
print(f"逐通道 scale 范围: [{scale_pc.min():.6f}, {scale_pc.max():.6f}]")
print(f"逐通道 scale 标准差: {scale_pc.std():.6f} (差异越大，逐通道优势越明显)")

### 逐组量化（Per-Group Quantization）

#### 什么是逐组量化？

逐组量化将权重按组（如128列一组）分别计算量化参数，是逐通道和逐张量之间的折中方案。每组共享一组scale，在精度和存储开销之间取得平衡。AWQ、GPTQ等主流LLM量化算法均采用此策略。

#### 为什么用逐组量化？

1. **精度优于逐张量**：组内权重的数值范围更相似，scale更精确，量化误差更低。
2. **开销低于逐通道**：逐通道需要$C_{out}$个scale，逐组仅需$\frac{C_{out} \times C_{in}}{g}$个scale（$g$为组大小），额外存储极小。
3. **LLM量化的标准做法**：group_size=128是LLM量化的主流配置，在INT4量化中几乎不增加额外存储。

#### 逐组量化的数学公式

将权重 $W \in \mathbb{R}^{m \times n}$ 按最后一维分为 $\frac{n}{g}$ 组，每组$g$个元素：

$$W_{grouped} = W.reshape(m \times \frac{n}{g}, g)$$

$$s_k = \frac{\max(|W_{grouped,k,:}|)}{2^{b-1}-1}, \quad k = 0, 1, ..., \frac{m \times n}{g}-1$$

$$W_{q,k,:} = \text{clamp}(\text{round}(W_{grouped,k,:} / s_k), q_{min}, q_{max})$$

其中：
- $g$：组大小（group_size），通常为128
- $s_k$：第$k$组的缩放因子
- $W_{grouped,k,:}$：第$k$组的$g$个元素
- 额外存储：$\frac{m \times n}{g}$个FP32 scale值，占比仅$\frac{32}{g \times b}$（如group_size=128, INT4时为6.25%）

In [ ]:
def per_group_quantize(tensor: torch.Tensor, n_bits: int = 4, group_size: int = 128) -> tuple:
    """逐组量化：将权重按group_size分组，每组独立计算scale"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    orig_shape = tensor.shape
    assert tensor.shape[-1] % group_size == 0, f"最后一维{tensor.shape[-1]}必须能被group_size{group_size}整除"
    tensor_grouped = tensor.reshape(-1, group_size)
    scale = tensor_grouped.abs().amax(dim=-1, keepdim=True) / q_max
    scale = torch.clamp(scale, min=1e-8)
    quantized = torch.clamp(torch.round(tensor_grouped / scale), q_min, q_max)
    return quantized.reshape(orig_shape), scale.reshape(-1)

def per_group_dequantize(quantized: torch.Tensor, scale: torch.Tensor, group_size: int = 128) -> torch.Tensor:
    """逐组反量化"""
    orig_shape = quantized.shape
    quantized_grouped = quantized.reshape(-1, group_size)
    scale_expanded = scale.unsqueeze(1).expand_as(quantized_grouped)
    deq = quantized_grouped.float() * scale_expanded
    return deq.reshape(orig_shape)

weight = torch.randn(128, 256)

q_g4, scale_g4 = per_group_quantize(weight, n_bits=4, group_size=128)
deq_g4 = per_group_dequantize(q_g4, scale_g4, group_size=128)

q_g8, scale_g8 = per_group_quantize(weight, n_bits=8, group_size=128)
deq_g8 = per_group_dequantize(q_g8, scale_g8, group_size=128)

q_pt4, scale_pt4 = symmetric_quantize(weight, n_bits=4)
deq_pt4 = symmetric_dequantize(q_pt4, scale_pt4)

print(f"=== 逐组量化 vs 逐张量量化 ===")
print(f"INT4 逐张量 MSE: {F.mse_loss(weight, deq_pt4):.6f}")
print(f"INT4 逐组(128) MSE: {F.mse_loss(weight, deq_g4):.6f}")
print(f"INT8 逐组(128) MSE: {F.mse_loss(weight, deq_g8):.6f}")
print(f"\n逐组量化额外存储开销:")
n_groups = weight.numel() // 128
print(f"  INT4 逐组 scale 参数量: {n_groups} (FP32), 占权重参数量{weight.numel()}的 {n_groups/weight.numel()*100:.2f}%")
print(f"  INT4 逐组总存储: {weight.numel() * 0.5 / 1024:.2f} KB (权重) + {n_groups * 4 / 1024:.4f} KB (scale)")

---
## 1.1.2 量化感知训练（Quantization-Aware Training, QAT）

### 什么是QAT？

量化感知训练（QAT）是在训练过程中插入伪量化（fake quantization）节点，模拟量化带来的误差，让模型在训练阶段就适应低精度表示，从而在真正量化后保持更高精度。与PTQ不同，QAT需要完整的训练流程和训练数据。

### 为什么用QAT？

1. **精度更高**：QAT通常比PTQ精度高1-3%，尤其在低比特（INT4及以下）量化时差距更明显。
2. **补偿量化误差**：训练过程中模型主动学习抵抗量化噪声，权重会自动调整到对量化友好的分布。
3. **低比特必需**：INT4/INT2量化时，PTQ精度损失过大，QAT几乎是唯一可行方案。

### QAT的原理

#### 伪量化（Fake Quantization）

伪量化在前向传播时模拟量化-反量化过程，引入量化噪声：

$$\hat{x} = \text{dequantize}(\text{quantize}(x)) = s \cdot \text{clamp}\left(\text{round}\left(\frac{x}{s}\right), q_{min}, q_{max}\right)$$

其中 $\hat{x}$ 是伪量化后的值，与 $x$ 的差异即为量化噪声。

#### 直通估计器（Straight-Through Estimator, STE）

量化函数的round操作不可微，STE假设梯度在量化范围内恒为1，超出范围为0：

$$\frac{\partial \hat{x}}{\partial x} \approx \begin{cases} 1 & \text{if } q_{min} \leq \text{round}(x/s) \leq q_{max} \\ 0 & \text{otherwise} \end{cases}$$

STE的有效性基于直觉：量化是微小的扰动，梯度方向在局部仍然有效。虽然不是精确梯度，但实践中效果很好。

其中：
- $\hat{x}$：伪量化后的浮点值
- $s$：缩放因子
- $q_{min}, q_{max}$：量化范围
- STE使得量化感知训练可以用标准的反向传播和优化器

In [ ]:
class FakeQuantizeSymmetric(torch.autograd.Function):
    """对称伪量化的前向和反向传播"""
    @staticmethod
    def forward(ctx, x, scale, n_bits=8):
        q_max = 2 ** (n_bits - 1) - 1
        q_min = -2 ** (n_bits - 1)
        x_q = torch.clamp(torch.round(x / scale), q_min, q_max)
        x_deq = x_q * scale
        ctx.save_for_backward(x, scale)
        ctx.n_bits = n_bits
        return x_deq

    @staticmethod
    def backward(ctx, grad_output):
        x, scale = ctx.saved_tensors
        q_max = 2 ** (ctx.n_bits - 1) - 1
        q_min = -2 ** (ctx.n_bits - 1)
        x_q = torch.round(x / scale)
        in_range = (x_q >= q_min) & (x_q <= q_max)
        grad_input = grad_output * in_range.float()
        return grad_input, None, None

class QATLinear(nn.Module):
    """带伪量化的线性层，用于QAT训练"""
    def __init__(self, in_features, out_features, n_bits=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.n_bits = n_bits
        self.weight_scale = None
        self.act_scale = None
        self._init_scale()

    def _init_scale(self):
        with torch.no_grad():
            self.weight_scale = self.linear.weight.abs().max() / (2 ** (self.n_bits - 1) - 1)
            self.weight_scale = torch.clamp(self.weight_scale, min=1e-8)
            self.act_scale = torch.tensor(6.0)

    def update_scale(self):
        with torch.no_grad():
            self.weight_scale = self.linear.weight.abs().max() / (2 ** (self.n_bits - 1) - 1)
            self.weight_scale = torch.clamp(self.weight_scale, min=1e-8)

    def forward(self, x):
        self.update_scale()
        w_fake_q = FakeQuantizeSymmetric.apply(self.linear.weight, self.weight_scale, self.n_bits)
        x_fake_q = FakeQuantizeSymmetric.apply(x, self.act_scale, self.n_bits)
        return F.linear(x_fake_q, w_fake_q, self.linear.bias)

class SimpleModel(nn.Module):
    def __init__(self, dim=64, n_bits=8):
        super().__init__()
        self.fc1 = QATLinear(dim, dim, n_bits)
        self.fc2 = QATLinear(dim, dim, n_bits)
        self.fc3 = QATLinear(dim, dim, n_bits)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def train_qat(model, train_data, epochs=50, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        x, y = train_data
        out = model(x)
        loss = F.mse_loss(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

dim = 64
x_train = torch.randn(256, dim)
y_train = torch.randn(256, dim)

model_fp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
model_qat = SimpleModel(dim=dim, n_bits=8)

with torch.no_grad():
    model_qat.fc1.linear.weight.copy_(model_fp[0].weight)
    model_qat.fc1.linear.bias.copy_(model_fp[0].bias)
    model_qat.fc2.linear.weight.copy_(model_fp[2].weight)
    model_qat.fc2.linear.bias.copy_(model_fp[2].bias)
    model_qat.fc3.linear.weight.copy_(model_fp[4].weight)
    model_qat.fc3.linear.bias.copy_(model_fp[4].bias)

losses = train_qat(model_qat, (x_train, y_train), epochs=100)

with torch.no_grad():
    out_fp = model_fp(x_train)
    out_qat = model_qat(x_train)
    fp_loss = F.mse_loss(out_fp, y_train)
    qat_loss = F.mse_loss(out_qat, y_train)

print(f"=== QAT训练效果 ===")
print(f"FP模型损失: {fp_loss:.6f}")
print(f"QAT模型损失: {qat_loss:.6f}")
print(f"QAT训练后损失下降: {(fp_loss - qat_loss)/fp_loss*100:.1f}%")
print(f"训练损失收敛趋势: {losses[0]:.4f} -> {losses[-1]:.4f}")

### 量化感知低秩微调（QLoRA风格）

#### 什么是QLoRA风格的QAT？

将基座模型保持量化状态（如NF4），仅对低秩适配器（LoRA）进行训练，是QAT与PEFT的结合。基座模型的量化权重冻结不更新，只有LoRA的少量参数参与训练。

#### 为什么用QLoRA风格的QAT？

1. **显存大幅降低**：基座模型以NF4（4bit）存储，7B模型仅需约3.5GB显存即可训练。
2. **训练成本低**：仅训练LoRA参数（通常<1%总参数），训练速度快。
3. **精度接近全量QAT**：LoRA适配器以高精度训练，能补偿量化误差。

#### 数学原理

线性层的前向计算：
$$y = x \cdot W_{deq}^T + x \cdot (A^T \cdot B^T) \cdot \frac{\alpha}{r}$$

其中：
- $W_{deq}$：NF4量化权重反量化后的FP16/FP32值，冻结不更新
- $A \in \mathbb{R}^{r \times d_{in}}$：LoRA下投影矩阵，可训练
- $B \in \mathbb{R}^{d_{out} \times r}$：LoRA上投影矩阵，可训练
- $r$：LoRA秩（通常4-64），远小于$d_{in}$和$d_{out}$
- $\alpha$：LoRA缩放因子，控制适配器的影响强度
- $\frac{\alpha}{r}$：缩放系数，确保不同rank下适配器的输出量级一致

In [ ]:
class NF4Quantizer:
    """NF4 (NormalFloat4) 量化器 - QLoRA使用的量化格式
    NF4假设权重服从正态分布，将量化级别均匀分布在正态分布的分位数上"""
    def __init__(self):
        self.quantile_values = self._compute_nf4_levels()

    def _compute_nf4_levels(self):
        _NF4_LEVELS = torch.tensor([
            -1.0, -0.6961928009986877, -0.5250734090805054, -0.39491748809814453,
            -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
            0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
            0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
            0.7229568362236023, 1.0
        ], dtype=torch.float32)
        return _NF4_LEVELS

    def quantize(self, tensor: torch.Tensor) -> tuple:
        abs_max = tensor.abs().amax()
        normalized = tensor / abs_max.clamp(min=1e-8)
        distances = (normalized.unsqueeze(-1) - self.quantile_values).abs()
        indices = distances.argmin(dim=-1).to(torch.uint8)
        return indices, abs_max

    def dequantize(self, indices: torch.Tensor, abs_max: torch.Tensor) -> torch.Tensor:
        return self.quantile_values[indices.long()] * abs_max

class LoRALayer(nn.Module):
    """低秩适配层"""
    def __init__(self, in_dim, out_dim, rank=8, alpha=16):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(rank, in_dim) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_dim, rank))

    def forward(self, x):
        return (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

class QLoRALinear(nn.Module):
    """QLoRA线性层：基座权重NF4量化 + LoRA适配器"""
    def __init__(self, in_features, out_features, rank=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.nf4 = NF4Quantizer()
        self.weight_indices = None
        self.weight_abs_max = None
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.lora = LoRALayer(in_features, out_features, rank=rank)

    def init_from_weight(self, weight: torch.Tensor):
        self.weight_indices, self.weight_abs_max = self.nf4.quantize(weight)
        self.weight_indices = nn.Parameter(self.weight_indices, requires_grad=False)
        self.weight_abs_max = nn.Parameter(self.weight_abs_max, requires_grad=False)

    def forward(self, x):
        w_deq = self.nf4.dequantize(self.weight_indices, self.weight_abs_max)
        base_out = F.linear(x, w_deq, self.bias)
        lora_out = self.lora(x)
        return base_out + lora_out

linear = nn.Linear(256, 128)
qlora_layer = QLoRALinear(256, 128, rank=8)
qlora_layer.init_from_weight(linear.weight.data)

x = torch.randn(4, 256)
with torch.no_grad():
    out_orig = linear(x)
    out_qlora = qlora_layer(x)

base_mem = 256 * 128 * 4 / 1024
qlora_weight_mem = 256 * 128 * 0.5 / 1024
qlora_lora_mem = (8 * 256 + 128 * 8) * 4 / 1024

print(f"=== QLoRA 内存分析 ===")
print(f"原始FP32权重: {base_mem:.2f} KB")
print(f"NF4量化权重: {qlora_weight_mem:.2f} KB (压缩{base_mem/qlora_weight_mem:.1f}x)")
print(f"LoRA参数: {qlora_lora_mem:.2f} KB (rank=8)")
print(f"QLoRA总计: {qlora_weight_mem + qlora_lora_mem:.2f} KB")
print(f"总压缩比: {base_mem/(qlora_weight_mem + qlora_lora_mem):.1f}x")
print(f"\n可训练参数: LoRA_A={8*256}, LoRA_B={128*8}, 总计={8*256+128*8}")
print(f"冻结参数: NF4权重索引={256*128}(4bit), abs_max=1(FP32)")

---
## 1.1.3 混合精度量化（Mixed-Precision Quantization）

### 什么是混合精度量化？

混合精度量化对不同层/模块使用不同的量化比特数。对量化敏感的层（如首层、末层、注意力层）保持较高精度（FP16/INT8），对不敏感的层（如中间FFN层）使用较低精度（INT4/INT2），在整体压缩率和精度之间取得最优平衡。

### 为什么用混合精度量化？

1. **层间敏感度差异大**：研究表明LLM各层对量化的敏感度差异可达10倍以上，统一比特数会造成敏感层精度损失或不敏感层存储浪费。
2. **最优精度-压缩权衡**：在相同平均比特数下，混合精度比统一精度精度更高；在相同精度下，混合精度压缩率更高。
3. **硬件友好**：端侧芯片通常支持多种精度（FP16/INT8/INT4），混合精度可充分利用硬件能力。

### 基于Hessian迹的敏感度分析

HAWQ-V2通过计算各层Hessian矩阵的迹（trace）来评估量化敏感度：

$$\text{Tr}(H_l) = \sum_{i} \lambda_i^{(l)}$$

其中 $H_l$ 是第$l$层损失函数对权重的Hessian矩阵，$\lambda_i^{(l)}$是其特征值。迹越大，该层对量化越敏感，应使用更高精度。

Hessian迹的Hutchinson近似：
$$\text{Tr}(H) \approx \frac{1}{N} \sum_{i=1}^{N} v_i^T H v_i$$

其中 $v_i$ 是随机向量（通常从Rademacher分布采样），$N$是采样次数。

参数含义：
- $H_l$：第$l$层权重的Hessian矩阵，维度为参数量×参数量
- $\lambda_i^{(l)}$：Hessian矩阵的第$i$个特征值
- $v_i$：随机扰动向量，用于Hutchinson方法估计迹
- $N$：采样次数，越多越精确但计算量越大

In [ ]:
def compute_hessian_trace_approx(layer, calib_data, n_samples=50):
    """使用Hutchinson方法近似计算Hessian对角线的迹"""
    trace_estimates = []
    params = list(layer.parameters())
    for _ in range(n_samples):
        v = [torch.randn_like(p) for p in params]
        loss = layer(calib_data).sum()
        grads = torch.autograd.grad(loss, params, create_graph=True)
        hv = torch.autograd.grad(grads, params, grad_outputs=v, retain_graph=False)
        trace = sum((h_i * v_i).sum().item() for h_i, v_i in zip(hv, v))
        trace_estimates.append(trace)
    return np.mean(trace_estimates)

class TransformerBlock(nn.Module):
    def __init__(self, dim=64, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x

class SmallTransformer(nn.Module):
    def __init__(self, n_layers=6, dim=64, n_heads=4):
        super().__init__()
        self.layers = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = SmallTransformer(n_layers=6, dim=64, n_heads=4)
calib_x = torch.randn(4, 16, 64, requires_grad=True)

sensitivities = []
for i, layer in enumerate(model.layers):
    trace = compute_hessian_trace_approx(layer, calib_x, n_samples=20)
    sensitivities.append(trace)
    print(f"Layer {i}: Hessian trace = {trace:.4f}")

sensitivities = np.array(sensitivities)
normalized = (sensitivities - sensitivities.min()) / (sensitivities.max() - sensitivities.min() + 1e-8)

bit_assignments = []
for s in normalized:
    if s > 0.7:
        bit_assignments.append(8)
    elif s > 0.3:
        bit_assignments.append(4)
    else:
        bit_assignments.append(2)

print(f"\n=== 混合精度分配结果 ===")
for i, (s, b) in enumerate(zip(normalized, bit_assignments)):
    print(f"Layer {i}: 敏感度={s:.4f} -> INT{b}")

total_bits = sum(b * sum(p.numel() for p in layer.parameters()) for b, layer in zip(bit_assignments, model.layers))
fp32_bits = sum(32 * sum(p.numel() for p in layer.parameters()) for layer in model.layers)
print(f"\n平均比特数: {total_bits/fp32_bits*32:.2f}")
print(f"整体压缩比: {fp32_bits/total_bits:.2f}x")

---
## 1.1.4 主流量化算法

### LLM.int8() - 混合精度分解

#### 什么是LLM.int8()？

LLM.int8()是首个实现LLM INT8推理几乎无损的方法。其核心思想是对矩阵乘法中的离群值特征维度使用FP16计算，其余维度使用INT8计算，从而在保持精度的同时获得INT8的效率。

#### 为什么LLM.int8()有效？

LLM推理时，激活值中存在极少数绝对值非常大的离群值（outlier），通常集中在特定的隐藏维度上（约0.1%的维度）。这些离群值对INT8量化极为敏感，但它们只占极小的比例。将这少量离群值用FP16精确计算，其余99.9%的值用INT8高效计算，即可兼顾精度和效率。

#### LLM.int8()的数学原理

矩阵乘法 $Y = XW^T$ 的混合精度分解：

$$Y = X_{normal} W_{normal}^T + X_{outlier} W_{outlier}^T$$

其中：
- $X_{normal}$：$X$中非离群值列组成的子矩阵，用INT8量化后计算
- $X_{outlier}$：$X$中离群值列组成的子矩阵，用FP16直接计算
- $W_{normal}, W_{outlier}$：$W$中对应列的子矩阵
- 离群值检测阈值：通常设为6.0，即$|x_{i,j}| > 6.0$的维度为离群值

**INT8子矩阵计算**：
$$X_{normal} W_{normal}^T \approx (X_{normal}^{int8} \cdot s_x) (W_{normal}^{int8} \cdot s_w)^T = s_x \cdot s_w^T \odot (X_{normal}^{int8} @ W_{normal}^{int8,T})$$

其中 $s_x, s_w$ 是INT8量化的scale向量，$\odot$ 表示外积。

In [ ]:
def llm_int8_matmul(hidden_states, weight, threshold=6.0):
    """LLM.int8() 混合精度矩阵乘法"""
    outlier_mask = hidden_states.abs() > threshold
    has_outlier = outlier_mask.any(dim=-1)
    if not has_outlier.any():
        scale = weight.abs().max() / 127.0
        w_int8 = torch.round(weight / scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        h_scale = hidden_states.abs().max() / 127.0
        h_int8 = torch.round(hidden_states / h_scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        result = h_int8.float() @ w_int8.float().T * (h_scale.unsqueeze(-1) * scale.unsqueeze(0))
        return result

    outlier_cols = outlier_mask.any(dim=0)
    normal_cols = ~outlier_cols

    result = torch.zeros(hidden_states.shape[0], weight.shape[0])

    if normal_cols.any():
        h_normal = hidden_states[:, normal_cols]
        w_normal = weight[:, normal_cols]
        scale_w = w_normal.abs().max() / 127.0
        w_int8 = torch.round(w_normal / scale_w.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        scale_h = h_normal.abs().max() / 127.0
        h_int8 = torch.round(h_normal / scale_h.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        result_int8 = h_int8.float() @ w_int8.float().T * (scale_h.unsqueeze(-1) * scale_w.unsqueeze(0))
        result += result_int8

    if outlier_cols.any():
        h_outlier = hidden_states[:, outlier_cols]
        w_outlier = weight[:, outlier_cols]
        result_fp16 = h_outlier.half() @ w_outlier.half().T
        result += result_fp16.float()

    return result

hidden = torch.randn(4, 512) * 3.0
hidden[:, 42] = torch.randn(4) * 20.0
hidden[:, 137] = torch.randn(4) * 15.0
weight = torch.randn(256, 512) * 0.1

result_fp32 = hidden @ weight.T
result_int8 = llm_int8_matmul(hidden, weight, threshold=6.0)

outlier_count = (hidden.abs() > 6.0).sum().item()
total_count = hidden.numel()

print(f"=== LLM.int8() 混合精度分解 ===")
print(f"离群值数量: {outlier_count}/{total_count} ({outlier_count/total_count*100:.2f}%)")
print(f"FP32结果范数: {result_fp32.norm():.4f}")
print(f"INT8混合结果范数: {result_int8.norm():.4f}")
print(f"相对误差: {(result_fp32 - result_int8).norm() / result_fp32.norm() * 100:.4f}%")
print(f"\n关键洞察: 仅{(outlier_count/total_count*100):.1f}%的离群值用FP16，其余99%+用INT8，精度几乎无损")

### GPTQ - 基于OBQ的逐层量化

#### 什么是GPTQ？

GPTQ是一种基于Optimal Brain Quantization（OBQ）框架的逐层后量化方法。它利用Hessian信息逐行量化权重，最小化每层的重建误差。GPTQ能在3-4bit量化下保持接近FP16的精度，是目前最流行的LLM权重量化方法之一。

#### 为什么GPTQ有效？

1. **利用二阶信息**：通过Hessian矩阵捕获权重间的相关性，量化一个权重时考虑对其他权重的影响，并做出补偿。
2. **逐行最优**：OBQ理论保证每行量化是局部最优的，GPTQ通过懒惰batch更新将OBQ的$O(n^4)$复杂度降至$O(n^3)$。
3. **逐组量化**：配合group_size=128的逐组量化，在精度和效率间取得平衡。

#### GPTQ的数学原理

GPTQ的核心是OBQ的逐权重量化与Hessian逆更新。对于权重矩阵$W \in \mathbb{R}^{m \times n}$：

**目标函数**：最小化量化引起的输出误差
$$\min_{\hat{W}} \|WX - \hat{W}X\|_2^2 = \min_{\hat{W}} \text{tr}((W - \hat{W})H(W - \hat{W})^T)$$

其中 $H = X^TX$ 是激活的Hessian矩阵。

**逐列量化与误差补偿**：量化第$i$列时，
$$w_q^{(i)} = \text{quant}(w^{(i)}), \quad \delta^{(i)} = w^{(i)} - w_q^{(i)}$$
$$W_{:,i+1:n} \leftarrow W_{:,i+1:n} - \frac{\delta^{(i)} H_{i,i+1:n}^{-1}}{H_{ii}^{-1}}$$

其中：
- $W$：当前层的权重矩阵，$m$为输出维度，$n$为输入维度
- $H = X^TX \in \mathbb{R}^{n \times n}$：激活协方差矩阵（Hessian）
- $H^{-1}$：Hessian逆矩阵，用于计算误差补偿方向
- $w^{(i)}$：第$i$列权重（量化前）
- $w_q^{(i)}$：第$i$列量化后的权重
- $\delta^{(i)}$：第$i$列的量化误差
- 误差补偿公式将当前列的量化误差按Hessian逆的比例分配到后续列
- block_size：懒惰更新的块大小，每处理block_size列才执行一次补偿更新

In [ ]:
def gptq_quantize_layer(W: torch.Tensor, H_inv: torch.Tensor, block_size=128, n_bits=4) -> tuple:
    """GPTQ简化实现：逐行量化权重
    W: [out_features, in_features] 权重矩阵
    H_inv: [in_features, in_features] Hessian逆矩阵
    """
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    W_q = W.clone()
    Err = torch.zeros_like(W)
    Q = torch.zeros_like(W)

    col_count = W.shape[1]
    for i1 in range(0, col_count, block_size):
        i2 = min(i1 + block_size, col_count)
        count = i2 - i1
        for i in range(i1, i2):
            w = W_q[:, i]
            d = torch.diag(H_inv)[i]
            scale = w.abs().max() / q_max
            scale = torch.clamp(scale, min=1e-8)
            q = torch.clamp(torch.round(w / scale), q_min, q_max)
            Q[:, i] = q * scale
            Err[:, i] = (w - Q[:, i]) / d
            W_q[:, i:] -= Err[:, i:i+1] * H_inv[i, i:].unsqueeze(0)

    return Q, W_q

out_features, in_features = 128, 256
W = torch.randn(out_features, in_features) * 0.1
X_calib = torch.randn(32, in_features)

H = (X_calib.T @ X_calib) / X_calib.shape[0]
H += 0.01 * torch.eye(in_features)
H_inv = torch.linalg.inv(H)

Q_4bit, W_updated = gptq_quantize_layer(W, H_inv, block_size=128, n_bits=4)

scale_pt = W.abs().max() / 7
Q_naive = torch.clamp(torch.round(W / scale_pt.clamp(min=1e-8)), -8, 7) * scale_pt

with torch.no_grad():
    X_test = torch.randn(64, in_features)
    Y_fp = X_test @ W.T
    Y_gptq = X_test @ Q_4bit.T
    Y_naive = X_test @ Q_naive.T

err_gptq = (Y_fp - Y_gptq).norm() / Y_fp.norm() * 100
err_naive = (Y_fp - Y_naive).norm() / Y_fp.norm() * 100

print(f"=== GPTQ vs 朴素量化 (INT4) ===")
print(f"朴素INT4输出相对误差: {err_naive:.4f}%")
print(f"GPTQ INT4输出相对误差: {err_gptq:.4f}%")
print(f"GPTQ误差降低: {(1 - err_gptq/err_naive)*100:.1f}%")
print(f"\nGPTQ核心优势: 利用Hessian信息补偿量化误差，逐行处理时考虑已量化列的影响")

### AWQ - 激活感知权重量化

#### 什么是AWQ？

AWQ（Activation-Aware Weight Quantization）是一种激活感知的权重量化方法。它发现并非所有权重同等重要——对激活分布影响最大的权重通道（salient weights）应被保护，通过缩放因子降低这些通道的量化误差。

#### 为什么AWQ有效？

1. **权重重要性不均**：AWQ观察到，权重通道的重要性取决于对应激活的大小。激活值大的通道，其权重的量化误差会被放大，对输出影响更大。
2. **缩放保护**：对重要通道的权重乘以缩放因子$s > 1$后再量化，等效于在重要通道上使用更小的量化步长，降低量化误差。
3. **无需反向传播**：AWQ仅需少量校准数据前向传播即可确定缩放因子，比QAT成本低得多。

#### AWQ的数学原理

对于线性层 $Y = XW^T$，AWQ寻找逐通道缩放因子$s$，使量化误差最小：

$$\min_{s} \|XW^T - X \cdot \text{Quant}(W \cdot \text{diag}(s))^T \cdot \text{diag}(s)^{-1}\|_F^2$$

等价于对权重先缩放再量化再反缩放：
$$\hat{W}_{:,j} = \text{Quant}(W_{:,j} \cdot s_j) / s_j$$

**Salient通道识别**：通过激活大小判断
$$\text{salient}_j = \begin{cases} \text{True} & \text{if } \max(|X_{:,j}|) > \text{median}(\max(|X_{:,k}|)) \\ \text{False} & \text{otherwise} \end{cases}$$

其中：
- $s_j$：第$j$个输入通道的缩放因子
- $W_{:,j}$：权重矩阵第$j$列（对应第$j$个输入通道）
- $X_{:,j}$：激活矩阵第$j$列
- $\text{Quant}(\cdot)$：量化函数
- salient通道：激活值大的通道，其权重需要保护
- 缩放因子搜索空间：$s \in \{0.5, 1.0, 2.0, 4.0, 8.0\}$，通过网格搜索选择最优值

In [ ]:
def awq_search_scale(W: torch.Tensor, X: torch.Tensor, n_bits=4, group_size=128) -> torch.Tensor:
    """AWQ缩放因子搜索
    W: [out_features, in_features]
    X: [n_samples, in_features] 校准数据
    """
    n_groups = W.shape[1] // group_size
    best_scales = torch.ones(W.shape[1])

    for g in range(n_groups):
        start = g * group_size
        end = start + group_size
        w_group = W[:, start:end]
        x_group = X[:, start:end]

        x_max = x_group.abs().amax(dim=0)
        salient_mask = x_max > x_max.median()

        scale_candidates = torch.tensor([0.5, 1.0, 2.0, 4.0, 8.0])
        best_loss = float('inf')
        best_scale = 1.0

        q_max = 2 ** (n_bits - 1) - 1
        q_min = -2 ** (n_bits - 1)

        for s in scale_candidates:
            w_scaled = w_group.clone()
            w_scaled[:, salient_mask] *= s
            scale_q = w_scaled.abs().amax() / q_max
            scale_q = torch.clamp(scale_q, min=1e-8)
            w_q = torch.clamp(torch.round(w_scaled / scale_q), q_min, q_max) * scale_q
            w_q[:, salient_mask] /= s

            loss = (x_group @ w_group.T - x_group @ w_q.T).norm().item()
            if loss < best_loss:
                best_loss = loss
                best_scale = s

        best_scales[start:end] = best_scale if salient_mask.any() else 1.0

    return best_scales

def awq_quantize(W: torch.Tensor, scales: torch.Tensor, n_bits=4) -> tuple:
    """AWQ量化：先缩放salient通道，再量化"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    W_scaled = W * scales.unsqueeze(0)
    scale = W_scaled.abs().amax() / q_max
    scale = torch.clamp(scale, min=1e-8)
    W_q = torch.clamp(torch.round(W_scaled / scale), q_min, q_max) * scale
    W_deq = W_q / scales.unsqueeze(0)
    return W_deq, scale

out_features, in_features = 128, 256
W = torch.randn(out_features, in_features) * 0.1
X_calib = torch.randn(32, in_features)

scales = awq_search_scale(W, X_calib, n_bits=4, group_size=128)
W_awq, _ = awq_quantize(W, scales, n_bits=4)

scale_naive = W.abs().max() / 7
W_naive = torch.clamp(torch.round(W / scale_naive.clamp(min=1e-8)), -8, 7) * scale_naive

X_test = torch.randn(64, in_features)
Y_fp = X_test @ W.T
Y_awq = X_test @ W_awq.T
Y_naive = X_test @ W_naive.T

err_awq = (Y_fp - Y_awq).norm() / Y_fp.norm() * 100
err_naive = (Y_fp - Y_naive).norm() / Y_fp.norm() * 100

print(f"=== AWQ vs 朴素量化 (INT4) ===")
print(f"朴素INT4输出相对误差: {err_naive:.4f}%")
print(f"AWQ INT4输出相对误差: {err_awq:.4f}%")
print(f"AWQ误差降低: {(1 - err_awq/err_naive)*100:.1f}%")
print(f"\nAWQ核心洞察: 保护对激活影响最大的权重通道(salient weights)，通过缩放因子降低这些通道的量化误差")

### SmoothQuant - 激活难度迁移到权重

#### 什么是SmoothQuant？

SmoothQuant是一种激活-权重联合量化方法，通过数学等价变换将激活中的离群值难度迁移到权重上，使得激活和权重都变得容易量化，从而实现W8A8（权重8bit+激活8bit）的高效推理。

#### 为什么SmoothQuant有效？

1. **激活离群值是量化瓶颈**：LLM中激活值存在极端离群值，使得激活量化误差巨大。而权重分布通常较平滑，有更多"余量"吸收难度。
2. **等价变换不改变输出**：逐通道缩放是数学等价变换，$Y = XW^T = (XS^{-1})(SW)^T$，输出完全不变。
3. **实现W8A8**：迁移后激活和权重都易于量化，使得INT8激活量化成为可能，在支持INT8矩阵乘法的硬件上可获得最大加速。

#### SmoothQuant的数学原理

核心变换：对每个输入通道$j$，将激活除以平滑因子$s_j$，权重乘以$s_j$：

$$Y = XW^T = (X \cdot S^{-1})(S \cdot W)^T$$

其中 $S = \text{diag}(s_1, s_2, ..., s_n)$ 是逐通道平滑因子的对角矩阵。

**平滑因子计算**：
$$s_j = \frac{\max(|X_{:,j}|)^\alpha}{\max(|W_{:,j}|)^{1-\alpha}}$$

其中：
- $s_j$：第$j$个通道的平滑因子
- $X_{:,j}$：校准数据中第$j$个通道的激活值
- $W_{:,j}$：权重矩阵第$j$列
- $\alpha$：迁移强度超参数，通常取0.5
  - $\alpha = 0$：不平滑，所有难度留在激活
  - $\alpha = 1$：全部迁移到权重
  - $\alpha = 0.5$：平衡迁移，实践中效果最好
- $\max(|X_{:,j}|)^\alpha$：激活难度度量，值越大说明该通道越难量化
- $\max(|W_{:,j}|)^{1-\alpha}$：权重承受能力度量，值越大说明权重越能吸收难度

**变换后的效果**：
$$\hat{x}_j = x_j / s_j, \quad \hat{w}_j = w_j \cdot s_j$$
激活值被缩小（离群值被抑制），权重值被放大（利用权重的量化余量）。两者都变得更易于INT8量化。

In [ ]:
def smooth_quant_smooth_factor(X: torch.Tensor, W: torch.Tensor, alpha=0.5) -> torch.Tensor:
    """计算SmoothQuant的逐通道平滑因子"""
    x_max = X.abs().amax(dim=0)
    w_max = W.abs().amax(dim=0)
    scales = (x_max.pow(alpha) / w_max.pow(1 - alpha)).clamp(min=1e-8)
    return scales

def smooth_quantize(W: torch.Tensor, X: torch.Tensor, alpha=0.5, n_bits=8) -> tuple:
    """SmoothQuant完整流程"""
    smooth_scales = smooth_quant_smooth_factor(X, W, alpha)
    X_smooth = X / smooth_scales.unsqueeze(0)
    W_smooth = W * smooth_scales.unsqueeze(0)

    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)

    w_scale = W_smooth.abs().amax() / q_max
    w_scale = torch.clamp(w_scale, min=1e-8)
    W_q = torch.clamp(torch.round(W_smooth / w_scale), q_min, q_max) * w_scale
    W_deq = W_q / smooth_scales.unsqueeze(0)

    x_scale = X_smooth.abs().amax() / q_max
    x_scale = torch.clamp(x_scale, min=1e-8)
    X_q = torch.clamp(torch.round(X_smooth / x_scale), q_min, q_max) * x_scale
    X_deq = X_q * smooth_scales.unsqueeze(0)

    return W_deq, X_deq, smooth_scales

out_features, in_features = 128, 256
W = torch.randn(out_features, in_features) * 0.1
X = torch.randn(32, in_features)
X[:, 42] = torch.randn(32) * 50.0
X[:, 137] = torch.randn(32) * 30.0

W_sq, X_sq, smooth_scales = smooth_quantize(W, X, alpha=0.5, n_bits=8)

print(f"=== SmoothQuant 激活难度迁移 ===")
print(f"原始激活范围: [{X.min():.2f}, {X.max():.2f}]")
print(f"平滑后激活范围: [{X_sq.min():.2f}, {X_sq.max():.2f}]")
print(f"激活最大值降低: {(X.abs().max() - X_sq.abs().max())/X.abs().max()*100:.1f}%")
print(f"\nSmooth因子 (离群值通道):")
print(f"  通道42: {smooth_scales[42]:.4f}")
print(f"  通道137: {smooth_scales[137]:.4f}")
print(f"  通道0 (正常): {smooth_scales[0]:.4f}")

X_test = torch.randn(64, in_features)
X_test[:, 42] = torch.randn(64) * 50.0
X_test[:, 137] = torch.randn(64) * 30.0

Y_fp = X_test @ W.T
Y_sq = X_test @ W_sq.T

w_scale_n = W.abs().max() / 127
W_naive = torch.clamp(torch.round(W / w_scale_n.clamp(min=1e-8)), -128, 127) * w_scale_n
Y_naive_w = X_test @ W_naive.T

err_sq = (Y_fp - Y_sq).norm() / Y_fp.norm() * 100
err_naive = (Y_fp - Y_naive_w).norm() / Y_fp.norm() * 100
print(f"\nW8A16精度对比:")
print(f"  朴素INT8权重误差: {err_naive:.4f}%")
print(f"  SmoothQuant W8误差: {err_sq:.4f}%")
print(f"\nSmoothQuant核心价值: 使激活也可量化(W8A8)，同时保持精度")

### 综合对比：各量化算法效果

下面对比不同量化方法在同一权重矩阵上的量化误差。关键观察：
1. 逐组量化显著优于逐张量量化，因为组内权重分布更均匀
2. GPTQ利用Hessian信息进行误差补偿，进一步降低量化误差
3. AWQ通过保护salient通道，在低比特下表现优异
4. 量化比特数越低，高级算法相比朴素量化的优势越明显

In [ ]:
def benchmark_quantization_methods():
    out_features, in_features = 256, 512
    W = torch.randn(out_features, in_features) * 0.1
    X_calib = torch.randn(64, in_features)
    X_test = torch.randn(128, in_features)
    Y_fp = X_test @ W.T

    results = {}

    scale_8 = W.abs().max() / 127
    W_int8 = torch.clamp(torch.round(W / scale_8.clamp(min=1e-8)), -128, 127) * scale_8
    results['INT8(朴素)'] = (X_test @ W_int8.T - Y_fp).norm() / Y_fp.norm() * 100

    scale_4 = W.abs().max() / 7
    W_int4 = torch.clamp(torch.round(W / scale_4.clamp(min=1e-8)), -8, 7) * scale_4
    results['INT4(朴素)'] = (X_test @ W_int4.T - Y_fp).norm() / Y_fp.norm() * 100

    q_g4, s_g4 = per_group_quantize(W, n_bits=4, group_size=128)
    W_g4 = per_group_dequantize(q_g4, s_g4, group_size=128)
    results['INT4(逐组)'] = (X_test @ W_g4.T - Y_fp).norm() / Y_fp.norm() * 100

    H = (X_calib.T @ X_calib) / X_calib.shape[0] + 0.01 * torch.eye(in_features)
    H_inv = torch.linalg.inv(H)
    Q_gptq, _ = gptq_quantize_layer(W, H_inv, block_size=128, n_bits=4)
    results['GPTQ(INT4)'] = (X_test @ Q_gptq.T - Y_fp).norm() / Y_fp.norm() * 100

    scales_awq = awq_search_scale(W, X_calib, n_bits=4, group_size=128)
    W_awq, _ = awq_quantize(W, scales_awq, n_bits=4)
    results['AWQ(INT4)'] = (X_test @ W_awq.T - Y_fp).norm() / Y_fp.norm() * 100

    print(f"{'方法':<20} {'相对误差(%)':<15} {'压缩比':<10}")
    print("-" * 45)
    for name, err in results.items():
        if 'INT8' in name:
            ratio = '4x'
        else:
            ratio = '8x'
        print(f"{name:<20} {err:<15.4f} {ratio:<10}")

    print(f"\n结论:")
    print(f"1. INT4比INT8误差大，但压缩比更高")
    print(f"2. 逐组量化显著优于逐张量量化")
    print(f"3. GPTQ/AWQ通过利用校准数据信息，进一步降低量化误差")

benchmark_quantization_methods()

### 产业级实战：使用 transformers + bitsandbytes 量化真实模型

在生产环境中，量化通常使用成熟的库而非手写实现。以下展示产业界标准做法：

- **bitsandbytes**：最广泛使用的LLM量化库，支持INT8/INT4/NF4量化。无需校准数据，加载时即量化，适合快速原型验证。
- **transformers BitsAndBytesConfig**：HuggingFace生态的标准量化接口，与`from_pretrained`无缝集成，一行代码即可量化加载模型。
- **GPTQ/AWQ**：需要 `auto-gptq` / `autoawq` 库（需GPU+编译），量化后模型可保存为量化格式，推理时直接加载，无需重新量化。

产业界量化方案选择指南：
| 场景 | 推荐方案 | 精度 | 速度 |
|------|---------|------|------|
| 端侧GPU部署 | AWQ/GPTQ W4A16 | 高 | 快 |
| 端侧CPU/NPU | SmoothQuant W8A8 | 中 | 最快 |
| 快速原型验证 | bitsandbytes NF4 | 中 | 中 |
| 极致压缩 | QuIP#/AQLM W2A16 | 低 | 快 |

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import time
import gc

MODEL_NAME = 'gpt2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
fp16_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map='auto' if torch.cuda.is_available() else None,
)
fp16_model.eval()

def measure_model(model, name, prompt='The future of AI is'):
    inputs = tokenizer(prompt, return_tensors='pt')
    if torch.cuda.is_available() and hasattr(model, 'device'):
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        start = time.perf_counter()
        outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        elapsed = time.perf_counter() - start
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    param_count = sum(p.numel() for p in model.parameters())
    mem_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024
    return {'name': name, 'params': param_count, 'mem_mb': mem_mb, 'latency_ms': elapsed * 1000, 'output': text}

fp16_result = measure_model(fp16_model, 'FP16')
print(f"=== FP16基线 ===")
print(f"参数量: {fp16_result['params']:,}")
print(f"内存: {fp16_result['mem_mb']:.1f} MB")
print(f"延迟: {fp16_result['latency_ms']:.1f} ms")
print(f"输出: {fp16_result['output']}")

del fp16_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
print(f"=== bitsandbytes INT8量化 ===")
int8_config = BitsAndBytesConfig(load_in_8bit=True)
int8_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=int8_config,
    device_map='auto' if torch.cuda.is_available() else None,
)
int8_model.eval()

int8_result = measure_model(int8_model, 'INT8')
print(f"参数量: {int8_result['params']:,}")
print(f"内存: {int8_result['mem_mb']:.1f} MB")
print(f"延迟: {int8_result['latency_ms']:.1f} ms")
print(f"输出: {int8_result['output']}")
print(f"内存节省: {(1 - int8_result['mem_mb']/fp16_result['mem_mb'])*100:.1f}%")

del int8_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
print(f"=== bitsandbytes NF4量化 (QLoRA格式) ===")
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
nf4_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=nf4_config,
    device_map='auto' if torch.cuda.is_available() else None,
)
nf4_model.eval()

nf4_result = measure_model(nf4_model, 'NF4')
print(f"参数量: {nf4_result['params']:,}")
print(f"内存: {nf4_result['mem_mb']:.1f} MB")
print(f"延迟: {nf4_result['latency_ms']:.1f} ms")
print(f"输出: {nf4_result['output']}")
print(f"内存节省: {(1 - nf4_result['mem_mb']/fp16_result['mem_mb'])*100:.1f}%")

print(f"\n=== 产业级量化对比 ===")
print(f"{'格式':<10} {'内存(MB)':<12} {'vs FP16':<12} {'输出一致':<10}")
print("-" * 44)
for r in [fp16_result, int8_result, nf4_result]:
    saving = f"{(1-r['mem_mb']/fp16_result['mem_mb'])*100:.0f}%" if r['name'] != 'FP16' else '-'
    print(f"{r['name']:<10} {r['mem_mb']:<12.1f} {saving:<12} {'-':<10}")

print(f"\n产业界最佳实践:")
print(f"1. 端侧GPU部署: AWQ/GPTQ W4A16 → 最小精度损失")
print(f"2. 端侧CPU/NPU: SmoothQuant W8A8 → 硬件友好")
print(f"3. 快速原型验证: bitsandbytes NF4 → 零校准数据")
print(f"4. 极致压缩: QuIP#/AQLM W2A16 → 2bit前沿")

del nf4_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()